# 🃏 Poker AI v4 — Kaggle Trainer

| GPU | Ingyenes idő | Mentés |
|---|---|---|
| T4 × 2 vagy P100 | **30 óra / hét** | `/kaggle/working/` → Output letöltés |

### Mielőtt elindítod:
- **Accelerator**: *Session Options → Accelerator → GPU T4 × 2*
- **Internet**: *Session Options → Internet → On*
- **Kód dataset**: a 2. lépés leírja hogyan töltsd fel

## Lépés 1 — Dependenciák

In [ ]:
import subprocess, torch

subprocess.run('pip install -q rlcard>=1.0.5 tensorboard>=2.13.0', shell=True)

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}  ({vram:.1f} GB)')

## Lépés 2 — Kódbázis betöltése

**Egyszeri előkészítés (Kaggle weboldalon, nem itt):**
1. Menj ide: [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new)
2. Töltsd fel a `poker_ai_v4.zip` fájlt
3. Név legyen: `poker-ai-v4-code` — állítsd **Private**-ra — kattints Create
4. Ebben a notebookban kattints: **+ Add Data → Your Datasets → poker-ai-v4-code**
5. Ezután `/kaggle/input/poker-ai-v4-code/` alatt lesz elérhető

*(Ha a zip neve más, módosítsd a `DATASET_NAME` változót lent)*

In [ ]:
import os, shutil, zipfile, glob

DATASET_NAME = 'poker-ai-v4-code'   # ← a Kaggle dataset neve
CODE_DIR     = '/kaggle/working/poker_ai'

# Ha már ki van bontva, kihagyjuk
if os.path.exists(os.path.join(CODE_DIR, 'train.py')):
    print(f'✅ Kód már megvan: {CODE_DIR}')
else:
    dataset_path = f'/kaggle/input/{DATASET_NAME}'

    if not os.path.exists(dataset_path):
        available = os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []
        raise FileNotFoundError(
            f'Dataset nem található: {dataset_path}\n'
            f'Elérhető datasetek: {available}\n\n'
            f'→ Kattints a + Add Data gombra és add hozzá: {DATASET_NAME}'
        )

    # Zip keresés
    zips = glob.glob(os.path.join(dataset_path, '*.zip'))
    if not zips:
        raise FileNotFoundError(
            f'Nincs .zip a datasetben. Tartalom: {os.listdir(dataset_path)}'
        )

    # Kibontás + projekt gyökér megkeresése
    tmp = '/kaggle/working/_tmp_extract'
    if os.path.exists(tmp): shutil.rmtree(tmp)
    with zipfile.ZipFile(zips[0], 'r') as z:
        z.extractall(tmp)

    found = None
    for root, dirs, files in os.walk(tmp):
        if 'train.py' in files:
            found = root
            break

    if found is None:
        raise RuntimeError('train.py nem található a zip-ben!')

    if os.path.exists(CODE_DIR): shutil.rmtree(CODE_DIR)
    shutil.copytree(found, CODE_DIR)
    shutil.rmtree(tmp)
    print(f'✅ Kód kibontva: {CODE_DIR}')

print(f'\nFájlok: {os.listdir(CODE_DIR)[:8]}')

## Lépés 3 — Patch alkalmazása

In [ ]:
import os

# ── PATCH 1: config.py — milestone_hands mező ─────────────────────────────
config_path = os.path.join(CODE_DIR, 'config.py')
with open(config_path, 'r', encoding='utf-8') as f:
    content = f.read()

if 'milestone_hands' not in content:
    patch = '''
    milestone_hands: int = 2000
    """Sanity teszt kézszám mérföldkőnél. Kaggle-on ajánlott: 500."""
'''
    anchor = '    milestone_dir_root: str = "ModellNaplo"'
    idx = content.find(anchor)
    if idx != -1:
        end = content.find('\n\n', idx)
        if end == -1: end = content.find('\n    # ──', idx)
        if end != -1:
            content = content[:end] + '\n' + patch + content[end:]
            with open(config_path, 'w', encoding='utf-8') as f:
                f.write(content)
            print('✅ config.py: milestone_hands hozzáadva')
        else: print('⚠️ config.py: beillesztési pont nem található')
    else: print('⚠️ config.py: anchor nem található')
else:
    print('ℹ️ config.py: milestone_hands már létezik')

# ── PATCH 2: runner.py — _milestone_str() fix ─────────────────────────────
runner_path = os.path.join(CODE_DIR, 'training', 'runner.py')
with open(runner_path, 'r', encoding='utf-8') as f:
    runner = f.read()

if '_milestone_str' not in runner:
    fn = '''
def _milestone_str(ep: int) -> str:
    """500_000 → '500k',  2_000_000 → '2M'"""
    return f"{ep // 1_000}k" if ep < 1_000_000 else f"{ep // 1_000_000}M"

'''
    idx = runner.find('def _run_milestone(')
    if idx != -1:
        runner = runner[:idx] + fn + runner[idx:]

    OLD = 'milestone_m = milestone_episodes // 1_000_000\n    milestone_str = f"{milestone_m}M"'
    if OLD in runner:
        runner = runner.replace(OLD, 'ms_str = _milestone_str(milestone_episodes)')
        fs = runner.find('def _run_milestone(')
        fe = runner.find('\ndef ', fs + 1)
        if fe == -1: fe = len(runner)
        runner = runner[:fs] + runner[fs:fe].replace('milestone_str', 'ms_str') + runner[fe:]
    print('✅ runner.py: _milestone_str() fix alkalmazva')
else:
    print('ℹ️ runner.py: fix már létezik')

if 'global MILESTONE_DIR_ROOT' not in runner:
    override = '''
    global MILESTONE_DIR_ROOT
    MILESTONE_DIR_ROOT = cfg.milestone_dir_root
'''
    anchor = 'session_start = time.time()'
    idx = runner.find(anchor)
    if idx != -1:
        runner = runner[:idx] + override + '    ' + runner[idx:]
        print('✅ runner.py: MILESTONE_DIR_ROOT override hozzáadva')

with open(runner_path, 'w', encoding='utf-8') as f:
    f.write(runner)

# ── PATCH 3: _train_session_cli.py — új argumentumok ──────────────────────
cli_path = os.path.join(CODE_DIR, '_train_session_cli.py')
with open(cli_path, 'r', encoding='utf-8') as f:
    cli = f.read()

if '--milestone-interval' not in cli:
    new_args = """
    parser.add_argument('--milestone-interval', type=int, default=None)
    parser.add_argument('--drive-output-dir', default=None)
"""
    drive_logic = """
    if args.drive_output_dir:
        output_base = args.drive_output_dir
        os.makedirs(output_base, exist_ok=True)
        mgr = ModelManager(output_base)
    else:
        output_base = BASE_DIR
    filtered["milestone_dir_root"] = mgr.tests_dir(args.model_name)
    if args.milestone_interval is not None:
        filtered["milestone_interval"] = args.milestone_interval
    mgr.ensure_model_dir(args.model_name, args.players)
"""
    idx = cli.find('args = parser.parse_args()')
    if idx != -1:
        cli = cli[:idx] + new_args + '\n    ' + cli[idx:]
    old = 'filtered["milestone_dir_root"] = mgr.tests_dir(args.model_name)'
    if old in cli:
        cli = cli.replace(old, drive_logic.strip())
    with open(cli_path, 'w', encoding='utf-8') as f:
        f.write(cli)
    print('✅ _train_session_cli.py: argumentumok hozzáadva')
else:
    print('ℹ️ _train_session_cli.py: argumentumok már léteznek')

print('\n✅ Minden patch kész!')

## Lépés 4 — ⚙️ Konfiguráció (ITT ÁLLÍTS!)

In [ ]:
import os, sys, torch

# ════════════════════════════════════════════════
MODEL_NAME         = '2max'          # modell neve
NUM_PLAYERS        = 2               # 2 / 6 / 9
EPISODES           = 2_000_000       # epizód ebben a sessionben
MILESTONE_INTERVAL = 500_000         # mérföldkő intervallum
MILESTONE_HANDS    = 500             # teszt kézszám — Kaggle-on tartsd alacsonyan!
TRAINING_STYLE     = 'exploitative'  # 'exploitative' | 'self_play' | 'aggressive'
NUM_ENVS           = 256             # T4-en 256, P100-on 512 is mehet
LEARNING_RATE      = 3e-4
# ════════════════════════════════════════════════

OUTPUT_BASE = '/kaggle/working/PokerAI_Models'
MODEL_DIR   = os.path.join(OUTPUT_BASE, 'models', MODEL_NAME)
PTH_PATH    = os.path.join(MODEL_DIR, f'{MODEL_NAME}_ppo_v4.pth')
TESTS_DIR   = os.path.join(MODEL_DIR, 'tests')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TESTS_DIR, exist_ok=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ModelManager inicializálás + config.json frissítés
from training.model_manager import ModelManager
mgr = ModelManager(OUTPUT_BASE)
mgr.ensure_model_dir(MODEL_NAME, NUM_PLAYERS)
cfg_data = mgr.load_config(MODEL_NAME)
cfg_data['config'].update({
    'milestone_interval':  MILESTONE_INTERVAL,
    'milestone_hands':     MILESTONE_HANDS,
    'num_envs':            NUM_ENVS,
    'learning_rate':       LEARNING_RATE,
    'training_style':      TRAINING_STYLE,
})
cfg_data['num_players'] = NUM_PLAYERS
mgr.save_config(MODEL_NAME, cfg_data)

# Checkpoint visszatöltés korábbi sessionből (ha van)
PREV_CHECKPOINT_DATASET = ''  # ← töltsd ki ha van pl. 'pokerai-2max-v1'
if PREV_CHECKPOINT_DATASET and not os.path.exists(PTH_PATH):
    import glob, zipfile
    ds = f'/kaggle/input/{PREV_CHECKPOINT_DATASET}'
    zips = glob.glob(os.path.join(ds, '*.zip'))
    pths = glob.glob(os.path.join(ds, '**', '*.pth'), recursive=True)
    if zips:
        with zipfile.ZipFile(zips[0], 'r') as z: z.extractall('/kaggle/working/')
        print(f'✅ Checkpoint zip visszatöltve: {zips[0]}')
    elif pths:
        import shutil
        shutil.copy2(pths[0], PTH_PATH)
        print(f'✅ Checkpoint visszatöltve: {PTH_PATH}')

# Összefoglaló
print('═' * 52)
print(f'  Modell:       {MODEL_NAME}  ({NUM_PLAYERS} játékos)')
print(f'  Epizódok:     {EPISODES:,}')
print(f'  Mérföldkő:    minden {MILESTONE_INTERVAL:,} ep')
print(f'  Teszt kéz:    {MILESTONE_HANDS}  ⚠️ ne adj meg sokat!')
print(f'  GPU envs:     {NUM_ENVS}')
print('─' * 52)
print(f'  Output:  {OUTPUT_BASE}')
print(f'  PTH:     {PTH_PATH}')
print('═' * 52)
if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    if isinstance(ck, dict) and 'episodes_trained' in ck:
        print(f'  ♻️  Folytatás: {ck["episodes_trained"]:,} epizódtól')
else:
    print(f'  🆕 Új tréning indul')

## Lépés 5 — 🚀 Tréning indítása

In [ ]:
import subprocess, sys, os, json

STYLE_POOLS = {
    'exploitative': {
        'fish':            {'enabled': True,  'weight': 0.8},
        'nit':             {'enabled': True,  'weight': 1.5},
        'calling_station': {'enabled': True,  'weight': 0.2},
        'lag':             {'enabled': True,  'weight': 1.5},
    },
    'self_play': {
        'fish':            {'enabled': False, 'weight': 0.0},
        'nit':             {'enabled': False, 'weight': 0.0},
        'calling_station': {'enabled': False, 'weight': 0.0},
        'lag':             {'enabled': False, 'weight': 0.0},
    },
    'aggressive': {
        'fish':            {'enabled': True,  'weight': 0.5},
        'nit':             {'enabled': True,  'weight': 2.5},
        'calling_station': {'enabled': False, 'weight': 0.0},
        'lag':             {'enabled': True,  'weight': 2.5},
    },
}

config_dict = {
    'num_envs':            NUM_ENVS,
    'buffer_collect_size': 2048,
    'hidden_size':         512,
    'learning_rate':       LEARNING_RATE,
    'milestone_hands':     MILESTONE_HANDS,
    'training_style':      TRAINING_STYLE,
    'bot_pool':            STYLE_POOLS.get(TRAINING_STYLE, STYLE_POOLS['exploitative']),
}

cmd = [
    sys.executable,
    os.path.join(CODE_DIR, '_train_session_cli.py'),
    '--model-name',         MODEL_NAME,
    '--pth-path',           PTH_PATH,
    '--players',            str(NUM_PLAYERS),
    '--episodes',           str(EPISODES),
    '--config-json',        json.dumps(config_dict),
    '--milestone-interval', str(MILESTONE_INTERVAL),
    '--drive-output-dir',   OUTPUT_BASE,
]

print(f'Indítás... (output: {OUTPUT_BASE})')
print('=' * 56)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, 'PYTHONPATH': CODE_DIR, 'PYTHONUNBUFFERED': '1'}
)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\n⚠️ Megszakítva — checkpoint megmarad, mentsd le (6. lépés)!')
    process.terminate()
    process.wait()

rc = process.returncode
print('\n✅ Kész!' if rc == 0 else f'\n❌ Hiba (kód: {rc})')
if rc == 0 or rc is None:
    print('→ Futtasd a 6. lépést a checkpoint mentéséhez!')

## Lépés 6 — 💾 Checkpoint becsomagolása és mentése

**Ezt minden session végén futtasd!** Utána a Kaggle *Save Version* (Commit) gombbal a `/kaggle/working/` tartalmát le tudod tölteni az Output szekcióból.

In [ ]:
import os, shutil, json, glob, sys

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ── Checkpoint állapot ────────────────────────────────────────────────────
print('═' * 52)
print('  CHECKPOINT ÁLLAPOT')
print('═' * 52)

if os.path.exists(PTH_PATH):
    from utils.checkpoint_utils import safe_load_checkpoint
    ck   = safe_load_checkpoint(PTH_PATH, map_location='cpu')
    ep   = ck.get('episodes_trained', 0) if isinstance(ck, dict) else 0
    t    = ck.get('time_spent', 0) if isinstance(ck, dict) else 0
    size = os.path.getsize(PTH_PATH) / 1e6
    print(f'  ✅ {PTH_PATH}')
    print(f'     Epizódok: {ep:,}  |  Tréning idő: {t/3600:.1f} óra  |  Méret: {size:.1f} MB')
else:
    print(f'  ❌ Nincs checkpoint: {PTH_PATH}')

# Mérföldkövek
ms_dirs = sorted(glob.glob(os.path.join(TESTS_DIR, '*/')), reverse=True)
print(f'\n  Mérföldkövek: {len(ms_dirs)} db')
for md in ms_dirs[:5]:
    name  = os.path.basename(md.rstrip('/'))
    grade = '?'
    for jf in glob.glob(os.path.join(md, '*.json')):
        try:
            with open(jf) as f: d = json.load(f)
            grade = d.get('summary', {}).get('grade', '?')
        except: pass
    pth_ok = '✅' if glob.glob(os.path.join(md, '*.pth')) else '❌'
    print(f'    {pth_ok} {name:<30}  grade: {grade}')

# ── Zip elkészítése ───────────────────────────────────────────────────────
zip_path = f'/kaggle/working/pokerai_{MODEL_NAME}_checkpoint'
shutil.make_archive(zip_path, 'zip', OUTPUT_BASE)
zip_size = os.path.getsize(zip_path + '.zip') / 1e6

print(f'\n  ✅ Zip elkészítve: {zip_path}.zip  ({zip_size:.1f} MB)')
print()
print('═' * 52)
print('  KÖVETKEZŐ LÉPÉSEK:')
print('═' * 52)
print('  1. Kattints: Save Version  (jobb felső sarok)')
print('  2. Mentés után: Notebook oldalán → Output')
print(f'  3. Töltsd le: pokerai_{MODEL_NAME}_checkpoint.zip')
print()
print('  Ha FOLYTATNI akarod a következő sessionben:')
print('  4. Töltsd fel a zip-et új Kaggle Datasetként')
print('  5. Add hozzá a notebookhoz (+ Add Data)')
print(f'  6. Írd be a dataset nevét PREV_CHECKPOINT_DATASET-be (4. lépés)')